# getting SNP and coverage data from PONmatrix
## python wrapper around samtools / mawktools
+ cleanpileup --> filterBED --> PON2SNP

In [ ]:
# get the code
import os

import numpy as np
import pandas as pd

# import package functions (main: sys.path.append('../code'); CNV_raw.addGCratio/addGenmap/PON2CNV)
import mycnv
from mycnv.raw import pon2cnv
from mycnv.tracks import add_gc, add_genmap
from mycnv.tools import cmd2df, mawk

# paths and parameters: ../config/config_dev.yaml via dev_config.py (main: hard-coded Dropbox paths)
from dev_config import load_dev
dev = load_dev()
# PON outputs go to <out_root>/PON/, never into the cached PON (main wrote into PON_path)
pon_out = dev["pon_out"]
fig_path = os.path.join(pon_out, "figures")
os.makedirs(fig_path, exist_ok=True)
print(f"mycnv {mycnv.__version__} ({os.path.dirname(mycnv.__file__)})")
print(f"PON {dev['pon_dir']} -> {pon_out}")

### get the config
+ `dev_config.load_dev()` reads `config/config_dev.yaml`: the PON folder, BED, GC and mappability tracks, and the
  Gen-2 CNV params (`mycnv.config.load_config`)
+ `pon_build.legacy_coords: true` reproduces main's bin labels/ExonPos (CHANGELOG.md)

In [ ]:
CNVconfig = dev["CNVconfig"]
pon_dir = dev["pon_dir"]          # reads matrix/{chrom}.pon.gz
bed_file = dev["bed"]
gc_path = dev["gc"]
genmap_path = dev["genmap"]
legacy_coords = dev["pon_build"].get("legacy_coords", False)
CNVconfig

## run the code

In [ ]:
chrom = "chr21"
c = CNVconfig["PONcoverage"]
pon_df, snp_df = pon2cnv(
    os.path.join(pon_dir, f"matrix/{chrom}.pon.gz"), bed_file, gc=gc_path, genmap=genmap_path, chrom=chrom,
    min_vaf=c["minVAF"], min_depth=c["minDepth"], min_cov=c["minCov"], legacy_coords=legacy_coords,
)
pon_df

##  the function

In [ ]:
# canonical code (was an inline copy of code/CNV_raw.py:PON2CNV): mycnv.raw.pon2cnv, which runs
# filterBed | PON2CNV (bash) and adds GC (mycnv.tracks.add_gc) and mappability (add_genmap); CLI: mycnv pon-coverage.
# This wrapper keeps the notebook's call and, like main, writes the PON SNP file snp/{chrom}.snp.gz (into pon_out).
def PON2CNV(
    chrom="",
    config={}
):
    c = config["PONcoverage"]
    matrix_file = os.path.join(pon_dir, f"matrix/{chrom}.pon.gz")
    if not os.path.isfile(matrix_file):
        print(f"PON matrix file {matrix_file} not found! Exiting.")
        return
    cov_df, snp_df = pon2cnv(
        matrix_file, bed_file, gc=gc_path, genmap=genmap_path, chrom=chrom,
        min_vaf=c["minVAF"], min_depth=c["minDepth"], min_cov=c["minCov"], legacy_coords=legacy_coords,
    )
    SNP_file = os.path.join(pon_out, f"snp/{chrom}.snp.gz")
    os.makedirs(os.path.dirname(SNP_file), exist_ok=True)
    print(f"Saving annotated heteroSNP data to {SNP_file}")
    snp_df.to_csv(SNP_file, index=False, sep="\t", compression="gzip")
    return cov_df, snp_df

In [ ]:
cov_df, snp_df = PON2CNV(
    chrom="chr21",
    config=CNVconfig
)
cov_df

In [ ]:
snp_df

### run all chroms
+ the chromosomes come from `pon_build.chroms` in config_dev.yaml (chr21 for the smoke test)

In [ ]:
chrom_list = dev["pon_build"].get("chroms") or [f"chr{c + 1}" for c in range(22)] + ['chrX']
for chrom in chrom_list:
    print(f"PON2CNV for {chrom}")
    cov_df, _ = PON2CNV(
    chrom=chrom,
    config=CNVconfig
    )
    PONcov_file = os.path.join(pon_out, f"cov/{chrom}.cov.gz")
    os.makedirs(os.path.dirname(PONcov_file), exist_ok=True)
    print(f"Writing PON coverage to {PONcov_file}")
    cov_df.to_csv(PONcov_file, sep="\t", index=False, compression="gzip")
print("FINISHED")